# Step 5: Inject Parent & Child Chunks into SQL

Inserts parent and child chunk metadata into the `documents` table in the SQL database.

This is critical for **two reasons**:
1. **Sparse (keyword) search** — The `HybridRetriever` runs BM25/tsvector keyword search against the `documents` table alongside Pinecone dense search, then fuses results with Reciprocal Rank Fusion (RRF).
2. **Parent chunk lookup** — After a child chunk is matched in Pinecone, the system uses `parent_chunk_id` to fetch the full parent text from SQL for richer LLM context.

**Input:** `chunks.json`  
**Output:** Rows inserted into `documents` table (SQLite in dev mode, PostgreSQL in production)

In [ ]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

from dotenv import load_dotenv
load_dotenv(dotenv_path=Path("../.env"))

INPUT_FILE = Path("./chunks.json")

In [ ]:
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    all_chunks = json.load(f)

parent_chunks = [c for c in all_chunks if c["is_parent"]]
child_chunks = [c for c in all_chunks if not c["is_parent"]]

print(f"Loaded {len(all_chunks)} chunks from {INPUT_FILE}")
print(f"  - {len(parent_chunks)} parent chunks (for sparse search + LLM context)")
print(f"  - {len(child_chunks)} child chunks (metadata linking to Pinecone vectors)")

In [ ]:
import asyncio
from config.settings import settings

print(f"Dev mode: {settings.dev_mode}")
print(f"Database: {settings.effective_database_url}")

In [ ]:
from app.models.database import async_engine, async_session_factory, init_db

async def setup_tables():
    if settings.dev_mode:
        await init_db()
        print("SQLite tables created / verified")
    else:
        print("PostgreSQL mode — tables should already exist (via init_db.sql)")

await setup_tables()

In [ ]:
from sqlalchemy import text as sql_text

async def inject_chunks_to_sql(parents, children):
    inserted_parents = 0
    inserted_children = 0
    skipped = 0

    async with async_session_factory() as session:
        for pc in parents:
            try:
                await session.execute(
                    sql_text("""
                        INSERT INTO documents (id, title, source_url, doc_type,
                                               tenant_id, content, chunk_index,
                                               token_count, embedding_id)
                        VALUES (:id, :title, :source_url, :doc_type, :tenant_id,
                                :content, :chunk_index, :token_count, :embedding_id)
                    """),
                    {
                        "id": pc["chunk_id"],
                        "title": pc["filename"],
                        "source_url": None,
                        "doc_type": "document",
                        "tenant_id": None,
                        "content": pc["text"],
                        "chunk_index": pc["chunk_index"],
                        "token_count": pc["token_count"],
                        "embedding_id": None,
                    },
                )
                inserted_parents += 1
            except Exception as e:
                if "UNIQUE" in str(e).upper() or "duplicate" in str(e).lower():
                    skipped += 1
                else:
                    raise

        for cc in children:
            try:
                await session.execute(
                    sql_text("""
                        INSERT INTO documents (id, title, source_url, doc_type,
                                               tenant_id, content, parent_chunk_id,
                                               chunk_index, token_count,
                                               embedding_id)
                        VALUES (:id, :title, :source_url, :doc_type, :tenant_id,
                                :content, :parent_chunk_id, :chunk_index,
                                :token_count, :embedding_id)
                    """),
                    {
                        "id": cc["chunk_id"],
                        "title": cc["filename"],
                        "source_url": None,
                        "doc_type": "document",
                        "tenant_id": None,
                        "content": cc["text"],
                        "parent_chunk_id": cc["parent_id"],
                        "chunk_index": cc["chunk_index"],
                        "token_count": cc["token_count"],
                        "embedding_id": cc["chunk_id"],
                    },
                )
                inserted_children += 1
            except Exception as e:
                if "UNIQUE" in str(e).upper() or "duplicate" in str(e).lower():
                    skipped += 1
                else:
                    raise

        await session.commit()

    return inserted_parents, inserted_children, skipped

parents_ok, children_ok, skipped = await inject_chunks_to_sql(parent_chunks, child_chunks)

print(f"Inserted {parents_ok} parent chunks (sparse search + context)")
print(f"Inserted {children_ok} child chunks (metadata with parent_id link)")
if skipped:
    print(f"Skipped {skipped} duplicates")

In [ ]:
async def verify_injection():
    async with async_session_factory() as session:
        total = await session.execute(sql_text("SELECT COUNT(*) FROM documents"))
        total_count = total.scalar()

        parents = await session.execute(
            sql_text("SELECT COUNT(*) FROM documents WHERE parent_chunk_id IS NULL")
        )
        parent_count = parents.scalar()

        children = await session.execute(
            sql_text("SELECT COUNT(*) FROM documents WHERE parent_chunk_id IS NOT NULL")
        )
        child_count = children.scalar()

        sample = await session.execute(
            sql_text("SELECT id, title, token_count, parent_chunk_id FROM documents LIMIT 5")
        )
        rows = sample.fetchall()

    print(f"Documents table summary:")
    print(f"  Total rows:    {total_count}")
    print(f"  Parent chunks: {parent_count} (searchable via keyword/BM25)")
    print(f"  Child chunks:  {child_count} (linked to Pinecone vectors)")
    print(f"\nSample rows:")
    for row in rows:
        role = "child" if row[3] else "parent"
        print(f"  [{role:6}] {row[1]} — {row[2]} tokens")

await verify_injection()

## How This Enables Hybrid Search

With both Pinecone and SQL populated, the `HybridRetriever` now performs:

1. **Dense search** (Pinecone) — embeds the query, finds nearest child chunk vectors
2. **Sparse search** (SQL) — runs keyword/BM25 search against the `content` column in `documents`
3. **Reciprocal Rank Fusion** — merges dense + sparse results, boosting chunks that appear in both
4. **Cross-encoder reranking** — final precision pass for top-K selection

Parent chunks in SQL serve double duty:
- **Sparse search targets** — keyword queries match against parent chunk text
- **Context expansion** — child matches are expanded to their parent via `parent_chunk_id`